### Executive KPI view

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_executive_kpis AS
SELECT
  'Historical sample' AS metric_scope,
  COUNT(*) AS total_trips,
  COUNT(DISTINCT pickup_date) AS active_days,
  ROUND(SUM(total_amount), 2) AS total_revenue,
  ROUND(AVG(total_amount), 2) AS avg_trip_amount,
  ROUND(AVG(trip_duration_minutes), 2) AS avg_trip_duration_minutes,
  ROUND(AVG(trip_distance_miles), 2) AS avg_trip_distance_miles
FROM nyc_taxi_company.silver.trips_clean;

### Top historical demand zones

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_top_demand_zones AS
SELECT
  pickup_borough,
  pickup_zone,
  ROUND(AVG(avg_trip_count), 2) AS avg_hourly_demand,
  ROUND(MAX(p90_trip_count), 2) AS p90_hourly_demand,
  ROUND(AVG(avg_total_amount), 2) AS avg_total_amount,
  ROUND(AVG(avg_trip_duration_minutes), 2) AS avg_trip_duration_minutes
FROM nyc_taxi_company.gold.zone_hour_demand_baseline
GROUP BY pickup_borough, pickup_zone;

### Weather Summary

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_weather_summary AS
SELECT
  weather_condition,
  pickup_borough,
  COUNT(*) AS zone_hour_rows,
  ROUND(AVG(trip_count), 2) AS avg_observed_trip_count,
  ROUND(AVG(baseline_avg_trip_count), 2) AS avg_baseline_trip_count,
  ROUND(AVG(demand_lift_pct), 2) AS avg_demand_lift_pct,
  ROUND(AVG(avg_trip_duration_minutes), 2) AS avg_trip_duration_minutes,
  ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph
FROM nyc_taxi_company.gold.weather_demand_impact
GROUP BY weather_condition, pickup_borough;

### Weather Sensitive Zones

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_weather_sensitive_zones AS
SELECT
  weather_condition,
  pickup_borough,
  pickup_zone,
  COUNT(*) AS observed_zone_hours,
  ROUND(AVG(demand_lift_pct), 2) AS avg_demand_lift_pct,
  ROUND(AVG(trip_count), 2) AS avg_observed_trip_count,
  ROUND(AVG(baseline_avg_trip_count), 2) AS avg_baseline_trip_count
FROM nyc_taxi_company.gold.weather_demand_impact
WHERE baseline_avg_trip_count >= 20
GROUP BY weather_condition, pickup_borough, pickup_zone;

### Sports Event Trend

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_sports_event_demand_trend AS
SELECT
  event_id,
  league,
  team_name,
  opponent_name,
  event_name,
  venue_name,
  venue_borough,
  event_start_ts_local,
  estimated_event_end_ts_local,
  bucket_start_ts,
  bucket_end_ts,
  relative_hour_from_start,
  event_phase,
  venue_catchment_zone,
  catchment_type,
  pickup_trip_count,
  ROUND(baseline_avg_trip_count, 2) AS baseline_avg_trip_count,
  ROUND(demand_lift_pct, 2) AS demand_lift_pct,
  demand_lift_band,
  ROUND(event_day_avg_hourly_trip_count, 2) AS event_day_avg_hourly_trip_count,
  ROUND(event_day_lift_vs_avg_hour_pct, 2) AS event_day_lift_vs_avg_hour_pct
FROM nyc_taxi_company.gold.sports_event_demand_trend;

### Sports Origin-Destination Flow

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_sports_event_od_flows AS
SELECT
  event_id,
  league,
  team_name,
  opponent_name,
  event_name,
  venue_name,
  venue_borough,
  event_start_ts_local,
  estimated_event_end_ts_local,
  flow_window,
  flow_direction,
  venue_catchment_zone,
  catchment_type,
  origin_borough,
  origin_zone,
  destination_borough,
  destination_zone,
  trip_count,
  ROUND(avg_total_amount, 2) AS avg_total_amount,
  ROUND(avg_trip_duration_minutes, 2) AS avg_trip_duration_minutes,
  ROUND(avg_trip_distance_miles, 2) AS avg_trip_distance_miles
FROM nyc_taxi_company.gold.sports_event_od_flows;

### Live Demand View

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_live_demand AS
SELECT
  window_start_ts,
  window_end_ts,
  pickup_location_id,
  pickup_borough,
  pickup_zone,
  live_request_count,
  baseline_avg_trip_count,
  demand_ratio,
  demand_lift_pct,
  anomaly_band
FROM nyc_taxi_company.streaming.demand_vs_baseline;

### Live Surge Alerts

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_live_surge_alerts AS
SELECT
  window_start_ts,
  window_end_ts,
  pickup_location_id,
  pickup_borough,
  pickup_zone,
  live_request_count,
  baseline_avg_trip_count,
  demand_ratio,
  ROUND((demand_ratio - 1) * 100, 2) AS demand_lift_pct,
  anomaly_band,
  related_event_id,
  related_event_name,
  related_venue_name,
  event_phase,
  weather_condition,
  alert_reason
FROM nyc_taxi_company.streaming.surge_alerts;

### Average Trips and Revenue by day of week across boroughs

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_dow_borough_demand AS
SELECT
  pickup_borough,
  pickup_day_of_week,
  CASE pickup_day_of_week
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END AS day_of_week_name,
  CASE pickup_day_of_week
    WHEN 1 THEN 7
    WHEN 2 THEN 1
    WHEN 3 THEN 2
    WHEN 4 THEN 3
    WHEN 5 THEN 4
    WHEN 6 THEN 5
    WHEN 7 THEN 6
  END AS day_of_week_sort,
  COUNT(*) AS total_trip_count,
  COUNT(DISTINCT pickup_date) AS observed_days,
  ROUND(COUNT(*) / COUNT(DISTINCT pickup_date), 2) AS avg_daily_trip_count,
  ROUND(SUM(total_amount) / COUNT(DISTINCT pickup_date), 2) AS avg_daily_revenue,
  ROUND(AVG(total_amount), 2) AS avg_trip_amount
FROM nyc_taxi_company.silver.trips_clean
GROUP BY
  pickup_borough,
  pickup_day_of_week;

### Event Operations Opportunity Matrix

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_event_ops_opportunity AS
WITH post_event_trend AS (
  SELECT
    venue_name,
    league,
    team_name,
    SUM(pickup_trip_count) AS post_event_pickups,
    AVG(demand_lift_pct) AS avg_post_event_demand_lift_pct,
    AVG(event_day_lift_vs_avg_hour_pct) AS avg_event_day_lift_pct
  FROM nyc_taxi_company.gold.sports_event_demand_trend
  WHERE event_phase = 'Post-event'
    AND catchment_type = 'primary'
  GROUP BY venue_name, league, team_name
),
post_event_flows AS (
  SELECT
    venue_name,
    SUM(trip_count) AS post_event_outbound_trips,
    SUM(trip_count * avg_total_amount) / NULLIF(SUM(trip_count), 0) AS weighted_avg_trip_amount,
    SUM(trip_count * avg_trip_duration_minutes) / NULLIF(SUM(trip_count), 0) AS weighted_avg_duration_minutes
  FROM nyc_taxi_company.gold.sports_event_od_flows
  WHERE flow_window = 'Post-event outbound'
    AND catchment_type = 'primary'
  GROUP BY venue_name
)
SELECT
  t.venue_name,
  t.league,
  t.team_name,
  t.post_event_pickups,
  ROUND(t.avg_post_event_demand_lift_pct, 2) AS avg_post_event_demand_lift_pct,
  ROUND(t.avg_event_day_lift_pct, 2) AS avg_event_day_lift_pct,
  f.post_event_outbound_trips,
  ROUND(f.weighted_avg_trip_amount, 2) AS weighted_avg_trip_amount,
  ROUND(f.weighted_avg_duration_minutes, 2) AS weighted_avg_duration_minutes,
  ROUND(f.post_event_outbound_trips * f.weighted_avg_trip_amount, 2) AS estimated_post_event_revenue,
  CASE
    WHEN t.avg_post_event_demand_lift_pct >= 50 AND f.post_event_outbound_trips >= 100 THEN 'High Priority'
    WHEN t.avg_post_event_demand_lift_pct >= 20 OR f.post_event_outbound_trips >= 100 THEN 'Medium Priority'
    ELSE 'Monitor'
  END AS ops_priority
FROM post_event_trend t
LEFT JOIN post_event_flows f
  ON t.venue_name = f.venue_name;

### Dimension views for slicers

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_dim_month AS
SELECT DISTINCT
  date_format(pickup_date, 'yyyy-MM') AS year_month,
  year(pickup_date) AS pickup_year,
  month(pickup_date) AS pickup_month,
  date_format(pickup_date, 'MMM yyyy') AS month_label
FROM nyc_taxi_company.silver.trips_clean
WHERE pickup_date IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_dim_borough AS
SELECT DISTINCT
  pickup_borough
FROM nyc_taxi_company.silver.trips_clean
WHERE pickup_borough IS NOT NULL;

### Page 1 KPIs

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_exec_daily_borough_metrics AS
SELECT
  pickup_date,
  date_format(pickup_date, 'yyyy-MM') AS year_month,
  year(pickup_date) AS pickup_year,
  month(pickup_date) AS pickup_month,
  pickup_borough,

  COUNT(*) AS trip_count,
  ROUND(SUM(total_amount), 2) AS total_revenue,
  ROUND(SUM(trip_distance_miles), 2) AS total_distance_miles,
  ROUND(SUM(trip_duration_minutes), 2) AS total_duration_minutes,

  ROUND(SUM(total_amount) / NULLIF(COUNT(*), 0), 2) AS avg_trip_amount,
  ROUND(SUM(trip_distance_miles) / NULLIF(COUNT(*), 0), 2) AS avg_trip_distance_miles,
  ROUND(SUM(trip_duration_minutes) / NULLIF(COUNT(*), 0), 2) AS avg_trip_duration_minutes

FROM nyc_taxi_company.silver.trips_clean
GROUP BY
  pickup_date,
  date_format(pickup_date, 'yyyy-MM'),
  year(pickup_date),
  month(pickup_date),
  pickup_borough;

### Top demand zones

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_top_demand_zones AS
WITH zone_hourly AS (
  SELECT
    date_format(pickup_date, 'yyyy-MM') AS year_month,
    pickup_borough,
    pickup_zone,
    pickup_location_id,
    pickup_date,
    pickup_hour,
    COUNT(*) AS hourly_trip_count,
    ROUND(AVG(total_amount), 2) AS avg_trip_amount
  FROM nyc_taxi_company.silver.trips_clean
  GROUP BY
    date_format(pickup_date, 'yyyy-MM'),
    pickup_borough,
    pickup_zone,
    pickup_location_id,
    pickup_date,
    pickup_hour
)
SELECT
  year_month,
  pickup_borough,
  pickup_zone,
  pickup_location_id,

  ROUND(AVG(hourly_trip_count), 2) AS avg_hourly_demand,
  percentile_approx(hourly_trip_count, 0.9) AS p90_hourly_demand,
  ROUND(AVG(avg_trip_amount), 2) AS avg_trip_amount

FROM zone_hourly
GROUP BY
  year_month,
  pickup_borough,
  pickup_zone,
  pickup_location_id;

### Weather demand lift by borough

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_weather_lift_borough AS
SELECT
  date_format(pickup_date, 'yyyy-MM') AS year_month,
  pickup_borough,
  weather_condition,

  COUNT(*) AS zone_hour_rows,
  ROUND(AVG(trip_count), 2) AS avg_observed_trip_count,
  ROUND(AVG(baseline_avg_trip_count), 2) AS avg_baseline_trip_count,
  ROUND(AVG(demand_lift_pct), 2) AS avg_demand_lift_pct,
  ROUND(AVG(avg_trip_duration_minutes), 2) AS avg_trip_duration_minutes,
  ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph

FROM nyc_taxi_company.gold.weather_demand_impact
WHERE pickup_date IS NOT NULL
GROUP BY
  date_format(pickup_date, 'yyyy-MM'),
  pickup_borough,
  weather_condition;

### Average daily trips by day of week

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_dow_borough_demand AS
SELECT
  date_format(pickup_date, 'yyyy-MM') AS year_month,
  pickup_borough,
  pickup_day_of_week,

  CASE pickup_day_of_week
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END AS day_of_week_name,

  CASE pickup_day_of_week
    WHEN 2 THEN 1
    WHEN 3 THEN 2
    WHEN 4 THEN 3
    WHEN 5 THEN 4
    WHEN 6 THEN 5
    WHEN 7 THEN 6
    WHEN 1 THEN 7
  END AS day_of_week_sort,

  COUNT(*) AS total_trip_count,
  COUNT(DISTINCT pickup_date) AS observed_days,

  ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT pickup_date), 0), 2) AS avg_daily_trip_count,
  ROUND(SUM(total_amount) / NULLIF(COUNT(DISTINCT pickup_date), 0), 2) AS avg_daily_revenue,
  ROUND(AVG(total_amount), 2) AS avg_trip_amount

FROM nyc_taxi_company.silver.trips_clean
GROUP BY
  date_format(pickup_date, 'yyyy-MM'),
  pickup_borough,
  pickup_day_of_week;

### Live demand passthrough view

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_live_demand AS
SELECT
  window_start_ts,
  window_end_ts,
  pickup_location_id,
  pickup_borough,
  pickup_zone,
  live_request_count,
  baseline_avg_trip_count,
  demand_ratio,
  ROUND((demand_ratio - 1) * 100, 2) AS demand_lift_pct,
  anomaly_band
FROM nyc_taxi_company.streaming.demand_vs_baseline;

### Live Zone Summary

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_live_zone_summary AS
SELECT
  pickup_location_id,
  pickup_borough,
  pickup_zone,

  SUM(live_request_count) AS live_request_count,
  SUM(baseline_avg_trip_count) AS baseline_avg_trip_count,

  ROUND(
    SUM(live_request_count) / NULLIF(SUM(baseline_avg_trip_count), 0),
    2
  ) AS demand_ratio,

  ROUND(
    (
      SUM(live_request_count) / NULLIF(SUM(baseline_avg_trip_count), 0) - 1
    ) * 100,
    2
  ) AS demand_lift_pct,

  MAX(demand_ratio) AS max_window_demand_ratio,

  CASE
    WHEN SUM(live_request_count) / NULLIF(SUM(baseline_avg_trip_count), 0) >= 2.0 THEN 'High'
    WHEN SUM(live_request_count) / NULLIF(SUM(baseline_avg_trip_count), 0) >= 1.3 THEN 'Medium'
    ELSE 'Normal'
  END AS anomaly_band

FROM nyc_taxi_company.gold.pbi_live_demand
GROUP BY
  pickup_location_id,
  pickup_borough,
  pickup_zone;

### Live surge alerts

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi_company.gold.pbi_live_surge_alerts AS
SELECT
  window_start_ts,
  window_end_ts,
  pickup_location_id,
  pickup_borough,
  pickup_zone,
  live_request_count,
  baseline_avg_trip_count,
  demand_ratio,
  ROUND((demand_ratio - 1) * 100, 2) AS demand_lift_pct,
  anomaly_band,
  related_event_id,
  related_event_name,
  related_venue_name,
  event_phase,
  weather_condition,
  alert_reason
FROM nyc_taxi_company.streaming.surge_alerts;